<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/7_Elasticsearch_BM25_Compras_Claras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# S07 · Elasticsearch Search Lab

**Propósito.** Construir un buscador textual que puedas explicar y repetir: Console primero, Python después.

**Regla de trabajo:** cada bloque importante indica qué hace, qué salida esperar, cómo verificar y qué revisar si falla.

## A · Modelo mental de la sesión

No pienses en Elasticsearch como una librería de Python. Piensa en tres interfaces para el mismo motor:

| Interfaz | Para qué sirve | Evidencia |
|---|---|---|
| Elastic UI | inspeccionar proyecto, índice y documentos | ves mapping, count y documentos |
| Console | hablar directamente con la API | request/response JSON |
| Python | automatizar lo que ya hiciste manualmente | bulk, búsqueda, evaluación |

La frase central de S07 es: **Python automatiza la misma API que primero entiendes en Console.**

## A1. Anatomía de una solicitud

**ELASTIC CONSOLE · NO SE EJECUTA EN COLAB**

Copia este tipo de bloque en Dev Tools / Console, no hagas clic sobre la ruta en Colab.

```text
GET mi_indice/_search
{
  "query": {
    "match": {
      "descripcion": "mantenimiento aeronaves"
    }
  }
}
```

- `GET` es el método.
- `mi_indice/_search` es recurso + operación.
- El JSON es el cuerpo de la solicitud.
- La respuesta traerá `hits.total`, `hits.hits`, `_id`, `_score` y `_source`.

**Error que evitamos:** Colab puede convertir rutas con `/` en enlaces azules. Esos enlaces no apuntan a Elasticsearch.

## A2. Patrones de búsqueda que aparecen en industria

| Producto | Necesidad textual | Campos que puntúan | Filtros estructurados | Hipótesis de ranking |
|---|---|---|---|---|
| E-commerce | zapatos de senderismo impermeables | `nombre^3`, `descripcion` | talla, stock, precio | el nombre expresa mejor intención |
| Empleo | data scientist NLP | `cargo^3`, `skills^2`, `descripcion` | país, remoto | cargo y skills concentran evidencia |
| Noticias | sobrecostos contratación | `titulo^4`, `subtitulo^2`, `cuerpo` | premium, fecha | el título indica centralidad temática |
| Soporte | error de autenticación token | `titulo^2`, `problema`, `solucion` | producto, versión | recuperar solución correcta primero |

La regla reusable: **necesidad textual → campos que puntúan → filtros → hipótesis → evaluación.**

# B · Preparar Python

En este bloque no estás aprendiendo Elasticsearch todavía; estás preparando datos y dependencias para automatizar más adelante.

**Debe aparecer:** versión de librerías y corpus cargado.

**Si falla:** revisa conexión de Colab o que el archivo remoto exista.

In [ ]:
!pip -q install elasticsearch pandas rank-bm25

In [ ]:
import json, math, re, unicodedata, os
from collections import Counter
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi

URL_S06 = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv'
corpus_raw = pd.read_csv(URL_S06)

columnas = ['id_proceso','nombre_proceso','descripcion','entidad','tipo_registro','url_secop']
for col in columnas:
    if col not in corpus_raw.columns:
        corpus_raw[col] = ''

corpus = (corpus_raw[columnas]
          .fillna('')
          .drop_duplicates('id_proceso')
          .reset_index(drop=True))
corpus['texto_busqueda'] = corpus['nombre_proceso'].astype(str) + ' ' + corpus['descripcion'].astype(str)

print('Filas fuente:', len(corpus_raw))
print('Procesos únicos:', len(corpus))
display(corpus.head(3))

## B1. Cómo leer el código anterior

- `pd.read_csv(URL_S06)` trae el producto de la sesión 6.
- `drop_duplicates('id_proceso')` evita indexar el mismo proceso varias veces.
- `texto_busqueda` junta nombre y descripción para la búsqueda local didáctica.

**Evidencia esperada:** cerca de 2.109 filas fuente y 1.994 procesos únicos.

**Error probable:** si el URL no carga, prueba abrirlo en el navegador; si cambió la ruta, no sigas con resultados parciales.

# C · Línea base local: literal vs ranking

Antes de usar Elastic, observa el problema: `contains()` filtra, pero no ordena bien relevancia.

In [ ]:
consulta = 'mantenimiento aeronaves'
mask = corpus['texto_busqueda'].str.lower().str.contains('mantenimiento', na=False)
print('Coincidencias literales con mantenimiento:', int(mask.sum()))
display(corpus.loc[mask, ['id_proceso','nombre_proceso','entidad']].head(10))

## C1. BM25 local didáctico

Este bloque no reemplaza Elasticsearch. Sirve para entender tres intuiciones:

1. frecuencia del término;
2. rareza en el corpus;
3. normalización por longitud del documento.

Después veremos que Elasticsearch tiene su propio analyzer, mapping y scoring.

In [ ]:
def normalizar(txt):
    txt = unicodedata.normalize('NFKD', str(txt)).encode('ascii','ignore').decode('ascii')
    return re.findall(r'[a-z0-9]+', txt.lower())

tokens = [normalizar(x) for x in corpus['texto_busqueda']]
bm25 = BM25Okapi(tokens)
scores = bm25.get_scores(normalizar(consulta))

local = corpus.copy()
local['score_local'] = scores
ranking_local = local.sort_values('score_local', ascending=False).head(5)
display(ranking_local[['id_proceso','score_local','nombre_proceso','entidad']])

# D · Console antes de Python

En Elastic Console ejecuta primero operaciones pequeñas. Esto te permite entender el motor sin depender de código.

## D1. Crear índice demo

**ELASTIC CONSOLE · NO SE EJECUTA EN COLAB**

```text
PUT s07-demo
{
  "mappings": {
    "properties": {
      "titulo": {"type": "text", "analyzer": "spanish"},
      "categoria": {"type": "keyword"}
    }
  }
}
```

**Debe aparecer:** `acknowledged: true`.

## D2. Inspeccionar analyzer

```text
POST s07-demo/_analyze
{
  "analyzer": "spanish",
  "text": "Servicios de mantenimiento de las aeronaves"
}
```

**Debes poder explicar:** al menos un token y por qué no coincide exactamente con la frase original.

# E · Conectar Python con Elasticsearch

Ahora sí automatizamos.

**Qué queremos hacer:** crear un cliente Python que pueda enviar solicitudes autenticadas al proyecto Elastic.

**Qué NO significa:** `client` no contiene los documentos; solo sabe hablar con el servicio.

**Errores típicos:**

| Síntoma | Qué revisar |
|---|---|
| 401 | API key |
| timeout | Project URL / red |
| 404 | nombre del índice |

In [ ]:
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

endpoint = input('Project URL / endpoint: ').strip()
api_key = getpass('API key: ').strip()

client = Elasticsearch(endpoint, api_key=api_key)
info = client.info()
print('Conexión verificada con:', info.get('cluster_name') or info.get('name'))

## E1. Índice personal y mapping

Cada estudiante/equipo usa un índice propio para no pisar el trabajo de los demás.

**Decisiones de mapping:**

- `nombre_proceso` y `descripcion`: `text` con analyzer `spanish`;
- `id_proceso`, `entidad`, `tipo_registro`: `keyword`;
- `url_secop`: `keyword`.

In [ ]:
ALIAS = input('Alias corto sin espacios: ').strip().lower() or 'demo'
ALIAS = re.sub(r'[^a-z0-9_-]+','-', ALIAS)
INDEX_NAME = f's07-compras-claras-{ALIAS}'

mappings = {
    'properties': {
        'id_proceso': {'type':'keyword'},
        'nombre_proceso': {'type':'text', 'analyzer':'spanish'},
        'descripcion': {'type':'text', 'analyzer':'spanish'},
        'entidad': {'type':'keyword'},
        'tipo_registro': {'type':'keyword'},
        'url_secop': {'type':'keyword'}
    }
}

if client.indices.exists(index=INDEX_NAME):
    print('El índice ya existe:', INDEX_NAME)
else:
    client.indices.create(index=INDEX_NAME, mappings=mappings)
    print('Índice creado:', INDEX_NAME)

In [ ]:
resp = client.indices.analyze(
    index=INDEX_NAME,
    analyzer='spanish',
    text='Servicios de mantenimiento de las aeronaves'
)
print([t['token'] for t in resp['tokens']])

# F · Ingesta con bulk y verificación

**Qué queremos hacer:** pasar de tres documentos manuales a 1.994 procesos.

Cada acción `bulk` tiene tres piezas esenciales:

- `_index`: a dónde va;
- `_id`: identificador estable;
- `_source`: documento JSON que se guarda.

**Regla profesional:** ejecutar no basta; debes verificar `errors` y `count()`.

In [ ]:
def acciones_bulk(df):
    for _, r in df.iterrows():
        yield {
            '_index': INDEX_NAME,
            '_id': str(r['id_proceso']),
            '_source': {
                'id_proceso': str(r['id_proceso']),
                'nombre_proceso': str(r['nombre_proceso']),
                'descripcion': str(r['descripcion']),
                'entidad': str(r['entidad']),
                'tipo_registro': str(r['tipo_registro']),
                'url_secop': str(r['url_secop'])
            }
        }

ok, errors = helpers.bulk(client, acciones_bulk(corpus), raise_on_error=False)
remote_count = client.count(index=INDEX_NAME)['count']
print('Acciones OK:', ok)
print('Errores bulk:', len(errors))
print('Conteo local:', len(corpus))
print('Conteo Elasticsearch:', remote_count)
print('¿Coinciden?:', len(corpus) == remote_count)

# G · Query DSL explicado

Construiremos la consulta por capas:

1. `match`: búsqueda textual en un campo;
2. `multi_match`: evidencia en varios campos;
3. boost: hipótesis de peso por campo;
4. `filter`: condición exacta que no suma score;
5. `highlight`: inspección de fragmentos coincidentes.

## G1. `match` vs `term`

- `match` analiza texto y sirve para full-text.
- `term` busca un valor exacto y sirve para `keyword`, IDs o categorías.

No uses `term` sobre un campo `text` esperando comportamiento de búsqueda textual.

In [ ]:
resp_match = client.search(
    index=INDEX_NAME,
    size=5,
    query={'match': {'descripcion': consulta}}
)

for h in resp_match['hits']['hits']:
    print(round(h['_score'],3), h['_id'], h['_source']['nombre_proceso'][:100])

## G2. `multi_match`, `filter` y `highlight`

La consulta siguiente dice:

- busca `mantenimiento aeronaves`;
- dale más peso a `nombre_proceso`;
- usa `descripcion` como evidencia secundaria;
- restringe a `historico_adjudicado`;
- devuelve fragmentos para inspección humana.

In [ ]:
query_B = {
    'bool': {
        'must': [{
            'multi_match': {
                'query': consulta,
                'fields': ['nombre_proceso^3','descripcion']
            }
        }],
        'filter': [{'term': {'tipo_registro': 'historico_adjudicado'}}]
    }
}

resp_B = client.search(
    index=INDEX_NAME,
    size=5,
    query=query_B,
    highlight={'fields': {'nombre_proceso': {}, 'descripcion': {}}}
)

rows=[]
for rank,h in enumerate(resp_B['hits']['hits'], start=1):
    rows.append({
        'rank': rank,
        'id_proceso': h['_id'],
        'score': h['_score'],
        'nombre_proceso': h['_source']['nombre_proceso'],
        'highlight': str(h.get('highlight',{}))[:250]
    })
tabla_B = pd.DataFrame(rows)
display(tabla_B)

# H · Evaluar relevancia con Precision@5

Antes de etiquetar, define un criterio observable.

Ejemplo: “relevante = el documento trata directamente mantenimiento de aeronaves, no solo menciona una aeronave de forma tangencial”.

Después etiqueta top 5 con 1/0.

`Precision@5 = relevantes en los cinco primeros / 5`

Esta lógica escala profesionalmente con `_rank_eval`, pero aquí la hacemos manual para entenderla.

In [ ]:
criterio = input('Criterio observable de relevancia: ').strip()
print('Criterio:', criterio)

etiquetas = []
for _, row in tabla_B.iterrows():
    print('
Rank', row['rank'], '|', row['nombre_proceso'][:160])
    val = input('¿Relevante? 1=sí, 0=no: ').strip()
    etiquetas.append(1 if val == '1' else 0)

p5 = sum(etiquetas) / 5
print('Precision@5:', p5)

# I · Reto de transferencia: Sala de redacción

Ahora cambia el dominio. Una editora busca antecedentes sobre **sobrecostos en contratación**.

Tendrás campos como:

- `titulo`;
- `subtitulo`;
- `cuerpo`;
- `categoria`;
- `premium`;
- `publicado`.

Decisiones esperadas:

- texto largo → `text`;
- categoría → `keyword`;
- premium → `boolean`;
- publicado → `date`;
- A: `titulo + subtitulo + cuerpo`;
- B: `titulo^4 + subtitulo^2 + cuerpo` + filtro `premium=false`.

No basta copiar: debes justificar mapping, query y evaluación.

In [ ]:
URL_NOTICIAS = 'https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_contratacion_2026.json'
try:
    noticias = pd.read_json(URL_NOTICIAS)
    print('Noticias cargadas:', len(noticias))
    display(noticias.head(3))
except Exception as e:
    print('No se pudo cargar el dataset de noticias desde la URL:', e)
    print('El reto puede completarse después de verificar la ruta del archivo.')

# J · Exportación final

La entrega debe demostrar proceso, no solo resultados. Incluye:

1. índice usado;
2. consulta;
3. top 5;
4. criterio de relevancia;
5. P@5;
6. falso positivo o resultado dudoso;
7. siguiente experimento.

In [ ]:
salida = {
    'index': INDEX_NAME,
    'consulta': consulta,
    'criterio': criterio,
    'precision_at_5': p5,
    'top5': tabla_B.to_dict(orient='records')
}

Path('s07_config_busqueda.json').write_text(json.dumps(salida, ensure_ascii=False, indent=2), encoding='utf-8')
tabla_B.to_csv('s07_resultados_busqueda.csv', index=False)

md = f'''# Hito S07 · Relevancia textual

- Índice: {INDEX_NAME}
- Consulta: {consulta}
- Criterio: {criterio}
- Precision@5: {p5}

## Límite
La relevancia textual no demuestra irregularidad, calidad contractual ni riesgo. Solo ordena documentos según la consulta y configuración usadas.
'''
Path('hito_s07_relevancia.md').write_text(md, encoding='utf-8')
print('Archivos creados: s07_config_busqueda.json, s07_resultados_busqueda.csv, hito_s07_relevancia.md')